# Synthetic: a loop over a folder of CSVs

The shape of round 23's r23s2: `d = pd.read_csv(f)` in a loop, a column added to `d`, the
frames appended to a list and concatenated. Before d8c9f31 `d`'s recorded files were merged
across iterations, so iteration k snapshotted all k files so far (865,265 hashes for 1,312
files in one cell); every statement in the loop is also stored, so per-save cost shows here.

```
python benchmarks/bench_notebook_overhead.py benchmarks/synthetic_folder_loop.ipynb --mode off
python benchmarks/bench_notebook_overhead.py benchmarks/synthetic_folder_loop.ipynb --mode cold
```

In [ ]:
# Inputs are generated once per machine, into the temp dir, and aged a day:
# real exports are old files, and cash treats a file written moments ago
# differently from one untouched since this morning.
import os, tempfile, time
from pathlib import Path

EXPORTS = Path(tempfile.gettempdir()) / "cash_bench_folder_loop"
N_FILES = 1000
if not EXPORTS.exists() or len(os.listdir(EXPORTS)) != N_FILES:
    EXPORTS.mkdir(exist_ok=True)
    day_ago = time.time() - 86_400
    for i in range(N_FILES):
        p = EXPORTS / "pos_{:04d}.csv".format(i)
        p.write_text("store,sku,qty,price\n" + "\n".join(f"S{i % 23},K{(i * j) % 509},{j % 7 + 1},{(j % 50) + 0.99}" for j in range(200)) + "\n")
        os.utime(p, (day_ago, day_ago))
print(N_FILES, "inputs in", EXPORTS)

In [ ]:
import glob
import pandas as pd

files = sorted(glob.glob(os.path.join(EXPORTS, '*.csv')))
parts = []
for f in files:
    d = pd.read_csv(f, dtype={'store': 'string', 'sku': 'string'})
    d['source_file'] = os.path.basename(f)
    parts.append(d)
raw = pd.concat(parts, ignore_index=True)
print(len(raw), 'rows from', len(files), 'files')

In [ ]:
sales = raw.drop_duplicates(subset=['store', 'sku', 'qty', 'price']).copy()
sales['value'] = sales['qty'] * sales['price']
by_store = sales.groupby('store', as_index=False)['value'].sum()
by_sku = sales.groupby('sku')['qty'].sum().sort_values(ascending=False)
print(len(sales), len(by_store), by_sku.index[0])